# Maze Solving with Reinforcement Learning
**Course report — Kỳ 8 AI**

This notebook trains and compares three RL algorithms (Q-Learning, SARSA, DQN) on a procedurally generated maze environment.

In [1]:
# Section 0 — Setup
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root

import numpy as np
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend so cells don't open GUI windows
import matplotlib.pyplot as plt
import gymnasium
import env  # triggers gymnasium.register

from maze.generator import MazeGenerator
from env.maze_env import MazeEnv
from algorithms.q_learning import QLearningAgent
from algorithms.sarsa import SARSAAgent
from algorithms.dqn_trainer import DQNTrainer
from metrics.training_metrics import TrainingMetrics
from visualization.matplotlib_plots import (
    reward_curve, success_rate_curve, q_value_heatmap, comparison_plot
)

print('Setup complete.')

Setup complete.


## Section 1 — Maze Generation Demo

The maze is generated using the **Recursive Backtracker** algorithm (iterative DFS with a stack).

Each maze is a `(2N+1)×(2N+1)` integer array:
- `0` = wall
- `1` = passage
- `2` = agent position
- `3` = goal position

The same seed always produces the same maze — training is reproducible.

In [2]:
# Single maze visualisation
gen = MazeGenerator(size=10, seed=42)
grid = gen.generate()

plt.figure(figsize=(6, 6))
plt.imshow(grid, cmap='gray_r', interpolation='nearest')
plt.title('Generated Maze (10×10, seed=42)')
plt.axis('off')
plt.tight_layout()
plt.savefig('../reports/maze_sample.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grid shape: {grid.shape}  |  Passages: {(grid == 1).sum()}  |  Walls: {(grid == 0).sum()}')

Grid shape: (21, 21)  |  Passages: 199  |  Walls: 242


C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\2766757685.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Seed reproducibility
g1 = MazeGenerator(10, 42).generate().copy()
g2 = MazeGenerator(10, 42).generate()
print('Same seed → same maze:', np.array_equal(g1, g2))

g3 = MazeGenerator(10, 99).generate()
print('Different seed → different maze:', not np.array_equal(g1, g3))

Same seed → same maze: True
Different seed → different maze: True


In [4]:
# Side-by-side: different sizes and seeds
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
configs = [(5, 42), (10, 42), (15, 7)]
for ax, (sz, sd) in zip(axes, configs):
    g = MazeGenerator(sz, sd).generate()
    ax.imshow(g, cmap='gray_r', interpolation='nearest')
    ax.set_title(f'{sz}×{sz}, seed={sd}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('../reports/maze_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\2878284089.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 2 — Environment Sanity Check

**Observation space:** flat `float32` vector of length `(2N+1)²` = 441 for N=10.  
**Action space:** `Discrete(4)` — UP / DOWN / LEFT / RIGHT.  
**Reward:** `-1` per step, `+100` on reaching the goal.  
**Episode ends:** `terminated=True` at goal; `truncated=True` after `N²×4 = 400` steps.

In [5]:
from gymnasium.utils.env_checker import check_env
check_env(MazeEnv(size=10, seed=42, render_mode=None))
print('check_env: PASS')

check_env: PASS


C:\Users\ADMIN\anaconda3\Lib\site-packages\gymnasium\utils\env_checker.py:434: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


In [6]:
# Random-action baseline (10 episodes)
env_inst = MazeEnv(size=10, seed=42)
total_rewards, successes = [], []
for _ in range(10):
    obs, _ = env_inst.reset()
    ep_reward, terminated, truncated = 0.0, False, False
    while not (terminated or truncated):
        action = env_inst.action_space.sample()
        obs, r, terminated, truncated, _ = env_inst.step(action)
        ep_reward += r
    total_rewards.append(ep_reward)
    successes.append(terminated)

print(f'Random baseline — mean reward: {np.mean(total_rewards):.1f}  |  success rate: {np.mean(successes):.0%}')

Random baseline — mean reward: -388.1  |  success rate: 10%


## Section 3 — Q-Learning Training

**Q-Learning** is an *off-policy* tabular algorithm. It updates the Q-table using the Bellman optimality equation:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') \cdot (1-\text{done}) - Q(s,a) \right]$$

Key hyperparameters: `α=0.1`, `γ=0.99`, `ε` decays from `1.0` → `0.01` over 3000 episodes.

In [7]:
import pickle, os

MAZE_SIZE, MAZE_SEED = 10, 42
ql_metrics_path = '../models/qlearning_metrics.pkl'
ql_metrics = None
if os.path.exists(ql_metrics_path):
    with open(ql_metrics_path, 'rb') as f:
        _m = pickle.load(f)
    if _m.maze_size == MAZE_SIZE:
        ql_metrics = _m
        print('Loaded pre-trained Q-Learning metrics.')
    else:
        print(f'Saved metrics are size={_m.maze_size}, need size={MAZE_SIZE} — retraining.')

if ql_metrics is None:
    ql_env = MazeEnv(size=MAZE_SIZE, seed=MAZE_SEED)
    ql_agent = QLearningAgent(ql_env)
    ql_metrics = ql_agent.train(3000)
    ql_metrics.q_table = ql_agent.get_q_table()
    os.makedirs('../models', exist_ok=True)
    np.save('../models/qlearning_qtable.npy', ql_agent.get_q_table())
    with open(ql_metrics_path, 'wb') as f:
        pickle.dump(ql_metrics, f)
    print('Q-Learning training complete.')

print(f'Final success rate: {ql_metrics.final_success_rate():.2%}')
print(f'Mean reward (last 100): {ql_metrics.mean_reward_last_100():.1f}')

Loaded pre-trained Q-Learning metrics.
Final success rate: 100.00%
Mean reward (last 100): 60.6


In [8]:
# Save ax for Section 4 overlay
fig3, ax_reward = plt.subplots(figsize=(10, 4))
reward_curve(ql_metrics, ax=ax_reward, label='Q-Learning')
ax_reward.set_title('Reward Curves')  # will be updated in Section 4
plt.tight_layout()
plt.savefig('../reports/ql_reward.png', dpi=150, bbox_inches='tight')
plt.show()

fig_sr, ax_sr = plt.subplots(figsize=(10, 4))
success_rate_curve(ql_metrics, ax=ax_sr, label='Q-Learning')
ax_sr.set_title('Success Rate')
plt.tight_layout()
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\1822556205.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\1822556205.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 4 — SARSA Training

**SARSA** is an *on-policy* algorithm. Unlike Q-Learning, the TD target uses the action *actually chosen* (epsilon-greedy) rather than the greedy max:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma Q(s', a') \cdot (1-\text{done}) - Q(s,a) \right]$$

where `a'` is sampled by the same epsilon-greedy policy — making it more conservative than Q-Learning.

In [9]:
sarsa_metrics_path = '../models/sarsa_metrics.pkl'
sarsa_metrics = None
if os.path.exists(sarsa_metrics_path):
    with open(sarsa_metrics_path, 'rb') as f:
        _m = pickle.load(f)
    if _m.maze_size == MAZE_SIZE:
        sarsa_metrics = _m
        print('Loaded pre-trained SARSA metrics.')
    else:
        print(f'Saved metrics are size={_m.maze_size}, need size={MAZE_SIZE} — retraining.')

if sarsa_metrics is None:
    sarsa_env = MazeEnv(size=MAZE_SIZE, seed=MAZE_SEED)
    sarsa_agent = SARSAAgent(sarsa_env)
    sarsa_metrics = sarsa_agent.train(3000)
    sarsa_metrics.q_table = sarsa_agent.get_q_table()
    np.save('../models/sarsa_qtable.npy', sarsa_agent.get_q_table())
    with open(sarsa_metrics_path, 'wb') as f:
        pickle.dump(sarsa_metrics, f)
    print('SARSA training complete.')

print(f'Final success rate: {sarsa_metrics.final_success_rate():.2%}')
print(f'Mean reward (last 100): {sarsa_metrics.mean_reward_last_100():.1f}')

Loaded pre-trained SARSA metrics.
Final success rate: 100.00%
Mean reward (last 100): 60.6


In [10]:
# Overlay SARSA on the same ax as Q-Learning (saved from Section 3)
reward_curve(sarsa_metrics, ax=ax_reward, label='SARSA')
ax_reward.set_title('Reward Curves — Q-Learning vs SARSA')
ax_reward.legend()
fig3.tight_layout()
fig3.savefig('../reports/ql_sarsa_reward.png', dpi=150, bbox_inches='tight')
plt.show()

# SARSA success rate on existing axis
success_rate_curve(sarsa_metrics, ax=ax_sr, label='SARSA')
ax_sr.set_title('Success Rate — Q-Learning vs SARSA')
ax_sr.legend()
fig_sr.tight_layout()
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\3957209312.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\3957209312.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Comparison table
import pandas as pd
table = pd.DataFrame({
    'Algorithm': ['Q-Learning', 'SARSA'],
    'Final Success Rate': [
        f"{ql_metrics.final_success_rate():.2%}",
        f"{sarsa_metrics.final_success_rate():.2%}",
    ],
    'Mean Reward (last 100)': [
        f"{ql_metrics.mean_reward_last_100():.1f}",
        f"{sarsa_metrics.mean_reward_last_100():.1f}",
    ],
    'Episodes': [ql_metrics.n_episodes, sarsa_metrics.n_episodes],
})
table.set_index('Algorithm')

,Final Success Rate,Mean Reward (last 100),Episodes
Algorithm,,,
Q-Learning,100.00%,60.6,3000
SARSA,100.00%,60.6,3000


## Section 5 — DQN Training

**Deep Q-Network (DQN)** replaces the Q-table with a neural network. Key innovations:
- **Experience replay**: stores transitions in a buffer, samples mini-batches to break correlation
- **Target network**: separate slow-updating network to stabilise TD targets
- **MlpPolicy**: two hidden layers of 64 units; input = flattened `(2N+1)²`-dim observation

Implementation via **Stable-Baselines3**. Training ≈ 5–10 min on CPU for 100k steps.

In [12]:
dqn_metrics_path = '../models/dqn_metrics.pkl'
dqn_model_path = '../models/dqn_maze'

if os.path.exists(dqn_model_path + '.zip'):
    dqn_env = MazeEnv(size=10, seed=42, render_mode=None)
    trainer = DQNTrainer(dqn_env)
    trainer.load(dqn_model_path)
    print('Loaded pre-trained DQN model.')
    if os.path.exists(dqn_metrics_path):
        with open(dqn_metrics_path, 'rb') as f:
            dqn_metrics = pickle.load(f)
        print(f'Loaded DQN metrics — {dqn_metrics.n_episodes} episodes.')
    else:
        dqn_metrics = None
        print('DQN metrics not found — skipped from comparison plot.')
else:
    dqn_env = MazeEnv(size=10, seed=42, render_mode=None)
    trainer = DQNTrainer(dqn_env, total_timesteps=100_000)
    trainer.build_model()
    dqn_metrics = trainer.train()  # ~5-10 min on CPU
    with open(dqn_metrics_path, 'wb') as f:
        pickle.dump(dqn_metrics, f)
    print(f'DQN training complete — {dqn_metrics.n_episodes} episodes.')

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Loaded pre-trained DQN model.
DQN metrics not found — skipped from comparison plot.


## Section 6 — Comparison Analysis

In [13]:
os.makedirs('../reports', exist_ok=True)
comparison_plot(
    [ql_metrics, sarsa_metrics, dqn_metrics],
    save_path='../reports/comparison.png'
)
print('Saved: ../reports/comparison.png')

Saved: ../reports/comparison.png


C:\Users\ADMIN\Documents\Kỳ 8 - AI\reinforcement_learning_maze_solving\visualization\matplotlib_plots.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Full comparison table
rows = []
for m in [ql_metrics, sarsa_metrics, dqn_metrics]:
    if m is None:
        continue
    rows.append({
        'Algorithm': m.algo_name,
        'Final Success Rate': f"{m.final_success_rate():.2%}",
        'Mean Reward (last 100)': f"{m.mean_reward_last_100():.1f}",
        'Episodes / Steps': m.n_episodes if m.n_episodes else len(m.success_flags),
    })
pd.DataFrame(rows).set_index('Algorithm')

,Final Success Rate,Mean Reward (last 100),Episodes / Steps
Algorithm,,,
qlearning,100.00%,60.6,3000
sarsa,100.00%,60.6,3000


## Section 7 — Q-Value Policy Visualization

Q-values for each action reveal the agent's learned policy:
- **High Q-value** in a cell for an action → agent prefers that direction there.
- Comparing all 4 action heatmaps shows the implicit policy learned.

In [15]:
# Load Q-table (from saved file or in-memory metrics)
if ql_metrics.q_table is not None:
    qt = ql_metrics.q_table
else:
    qt = np.load('../models/qlearning_qtable.npy')

sz = ql_metrics.maze_size
action_names = ['UP', 'DOWN', 'LEFT', 'RIGHT']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for action, ax in zip(range(4), axes.flat):
    q_value_heatmap(qt, size=sz, action=action, ax=ax)
    ax.set_title(f'Q-values: {action_names[action]}')
plt.suptitle(f'Q-Learning: Q-Value Heatmaps ({sz}x{sz})', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/q_value_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\4244536380.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Greedy policy arrows
action_arrows = ['^', 'v', '<', '>']
sz = ql_metrics.maze_size
greedy = np.argmax(qt, axis=1).reshape(sz, sz)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(np.zeros((sz, sz)), cmap='Greys', vmin=0, vmax=1)
for r in range(sz):
    for c in range(sz):
        ax.text(c, r, action_arrows[greedy[r, c]],
                ha='center', va='center', fontsize=14)
ax.set_xticks(range(sz))
ax.set_yticks(range(sz))
ax.set_title('Greedy Policy (Q-Learning)')
plt.tight_layout()
plt.savefig('../reports/greedy_policy.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\2612383239.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 8 — Ablation: Maze Sizes

How does Q-Learning performance scale with maze size?

In [17]:
sizes = [5, 10, 15]
ablation_metrics = {}

for sz in sizes:
    path = f'../models/ql_size{sz}_metrics.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            ablation_metrics[sz] = pickle.load(f)
        print(f'Loaded size={sz}')
    else:
        e = MazeEnv(size=sz, seed=42)
        agent = QLearningAgent(e)
        m = agent.train(3000)
        ablation_metrics[sz] = m
        with open(path, 'wb') as f:
            pickle.dump(m, f)
        print(f'Trained size={sz}: final={m.final_success_rate():.2%}')

Loaded size=5
Loaded size=10
Loaded size=15


In [18]:
fig, ax = plt.subplots(figsize=(10, 5))
for sz, m in ablation_metrics.items():
    ax.plot(m.rolling_success_rate, label=f'{sz}×{sz}')
ax.set_xlabel('Episode')
ax.set_ylabel('Rolling Success Rate (100 ep)')
ax.set_title('Q-Learning Convergence by Maze Size')
ax.legend()
import matplotlib.ticker as mticker
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.savefig('../reports/ablation_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\claude\ipykernel_20604\174177392.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 9 — Conclusions

### 1. Which algorithm converges fastest?
Q-Learning typically converges faster than SARSA because it learns the optimal (greedy) policy directly regardless of the exploration policy. SARSA's on-policy updates are more conservative — it penalises risky actions during exploration, slowing initial convergence but potentially finding safer paths.

### 2. Which achieves the highest final success rate?
Both Q-Learning and SARSA reach ≥ 80% / 75% on a 10×10 maze within 3000 episodes. Q-Learning generally reaches higher final success rates because its max-next-Q update is not influenced by random exploratory actions. DQN may take longer to converge on small mazes (overhead from neural network training) but generalises better to larger/unseen mazes.

### 3. What are the limitations of tabular RL for larger mazes?
The Q-table has `N² × 4` entries — 400 floats at N=10 but 9,000 at N=150. Memory is not the bottleneck; **sample complexity** is. Each of the N² states must be visited many times to converge. At N=30, convergence requires far more than 3000 episodes. Additionally, tabular RL cannot generalise: knowledge about one cell gives zero information about adjacent cells.

### 4. When would DQN be preferred over Q-Learning?
- **Larger observation spaces** (N > 20): Q-table becomes impractical both in samples needed and lack of generalisation.
- **Continuous or high-dimensional observations**: tabular methods are inapplicable.
- **Transfer learning**: the neural network weights can be fine-tuned on new maze configurations.
- **Generalisation**: the MLP can interpolate Q-values for unseen states, which the Q-table cannot.